In [ ]:
"""
Please download all necessary packages below in a new conda environment:
conda install numpy pandas scikit-learn
conda install matplotlib jupyterlab
conda install pytorch torchvision torchaudio -c pytorch  # For neural network
# Or use pip: pip install torch
"""

import pandas as pd
import numpy as np



# ============================================================
# 0. Load local CSV: 'train.csv'
#    Assumes last column is 'critical_temp' or a column with that name.
# ============================================================
data_path = "train.csv"  # <- change to your actual path if needed
df = pd.read_csv(data_path)

# If the target column is not named 'critical_temp', adjust here:
if "critical_temp" in df.columns:
    target_col = "critical_temp"
else:
    # assume last column is the target
    target_col = df.columns[-1]
    print(f"WARNING: 'critical_temp' not found; using last column '{target_col}' as target.")

X = df.drop(columns=[target_col])
y = df[target_col]

print("=== Full dataset loaded from train.csv ===")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("First 5 feature names:", list(X.columns[:5]))
print()

In [ ]:
# Step 1: temp (85%) + verification (15%)
from sklearn.model_selection import train_test_split


X_temp, X_verif, y_temp, y_verif = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42
)

# Step 2: train + val from temp
val_fraction_within_temp = 0.15 / 0.85  # ≈ 0.17647 => overall 15% for val

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=val_fraction_within_temp,
    random_state=42
)

n_total = X.shape[0]
print("=== Split summary (70/15/15) ===")
print(f"Total samples:       {n_total}")
print(f"Train size:          {X_train.shape[0]}  ({X_train.shape[0]/n_total:.3f})")
print(f"Validation size:     {X_val.shape[0]}    ({X_val.shape[0]/n_total:.3f})")
print(f"Verification size:   {X_verif.shape[0]}  ({X_verif.shape[0]/n_total:.3f})")
print()


feat_names = X_train.columns  # all feature names

In [ ]:
# ============================================================
# Helper: map feature name -> human-readable English description
# ============================================================
def describe_feature(name: str) -> str:
    # Special case
    if name == "number_of_elements":
        return "Number of distinct chemical elements present in the compound."

    # Map property token to a human-readable phrase
    prop_map = {
        "atomic_mass": "atomic mass of constituent elements",
        "fie": "first ionization energy of constituent elements",
        "atomic_radius": "atomic radius of constituent elements",
        "Density": "mass density of constituent elements",
        "ElectronAffinity": "electron affinity of constituent elements",
        "FusionHeat": "heat of fusion (fusion enthalpy) of constituent elements",
        "ThermalConductivity": "thermal conductivity of constituent elements",
        "Valence": "valence electron count of constituent elements",
    }

    # Identify prefix and property name
    prefix = None
    prop_token = None
    for pre in [
        "wtd_mean_", "mean_",
        "wtd_gmean_", "gmean_",
        "wtd_entropy_", "entropy_",
        "wtd_range_", "range_",
        "wtd_std_", "std_",
    ]:
        if name.startswith(pre):
            prefix = pre
            prop_token = name[len(pre):]
            break

    # Fallback: generic description
    if prefix is None or prop_token is None:
        return f"Descriptor derived from elemental property '{name}', computed over the compound's composition."

    prop_phrase = prop_map.get(
        prop_token,
        f"property '{prop_token}' of constituent elements"
    )

    # Construct description according to prefix
    if prefix == "mean_":
        return f"Unweighted arithmetic mean of the {prop_phrase} in the compound."
    if prefix == "wtd_mean_":
        return f"Atomic-fraction-weighted arithmetic mean of the {prop_phrase} in the compound."
    if prefix == "gmean_":
        return f"Unweighted geometric mean of the {prop_phrase} in the compound."
    if prefix == "wtd_gmean_":
        return f"Atomic-fraction-weighted geometric mean of the {prop_phrase} in the compound."
    if prefix == "entropy_":
        return f"Entropy-like measure of how heterogeneous the {prop_phrase} are across different elements in the compound (higher = more diverse)."
    if prefix == "wtd_entropy_":
        return f"Atomic-fraction-weighted entropy-like measure of how heterogeneous the {prop_phrase} are across different elements in the compound."
    if prefix == "range_":
        return f"Range (max minus min) of the {prop_phrase} among the elements in the compound."
    if prefix == "wtd_range_":
        return f"Range-like measure of the {prop_phrase}, taking stoichiometric fractions into account."
    if prefix == "std_":
        return f"Standard deviation of the {prop_phrase} among the elements in the compound."
    if prefix == "wtd_std_":
        return f"Atomic-fraction-weighted standard deviation of the {prop_phrase} among the elements in the compound."

    # Generic fallback
    return f"Descriptor derived from elemental property '{prop_token}', computed over the compound's composition."

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, r2_score



# 2.1 Pearson correlation on TRAIN
df_train = X_train.copy()
df_train["critical_temp"] = y_train

corr_series = df_train.corr()["critical_temp"].drop("critical_temp")
corr_sorted = corr_series.reindex(corr_series.abs().sort_values(ascending=False).index)

print("=== Correlation stats (train) ===")
print("Max |corr|:", corr_sorted.abs().max())
print("Min |corr|:", corr_sorted.abs().min())
print("Top-10 by |corr|:\n", corr_sorted.head(10))
print()

top20_corr = set(corr_sorted.head(20).index)
top40_corr = set(corr_sorted.head(40).index)

In [ ]:
# 2.2 Random Forest
rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

rf_importances = rf.feature_importances_

# RF performance on VAL
y_val_pred = rf.predict(X_val)
mse_rf = mean_squared_error(y_val, y_val_pred)
rmse_rf = mse_rf ** 0.5
r2_rf = r2_score(y_val, y_val_pred)

print("=== Random Forest performance on validation set ===")
print(f"Val MSE:  {mse_rf:.4f}")
print(f"Val RMSE: {rmse_rf:.4f}")
print(f"Val R^2:  {r2_rf:.4f}")

# RF importance stats
rf_min = rf_importances.min()
rf_max = rf_importances.max()
rf_zero_count = np.sum(rf_importances == 0.0)
rf_nonzero_count = np.sum(rf_importances > 0.0)

print("=== Random Forest feature_importances_ stats ===")
print("Min importance:", rf_min)
print("Max importance:", rf_max)
print("Number of zero importances:", rf_zero_count)
print("Number of non-zero importances:", rf_nonzero_count)
print("Sum of importances (should be 1.0):", rf_importances.sum())

rf_imp_df = (
    pd.DataFrame({"feature": feat_names, "rf_importance": rf_importances})
    .sort_values("rf_importance", ascending=False)
)

print("Top-10 features by RF importance:\n", rf_imp_df.head(10))
print("\nBottom-10 features by RF importance:\n", rf_imp_df.tail(10))
print()

top20_rf = set(rf_imp_df["feature"].head(20))
top40_rf = set(rf_imp_df["feature"].head(40))


In [ ]:
# 2.3 Permutation importance on VAL
perm_result = permutation_importance(
    rf,
    X_val,
    y_val,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

perm_means = perm_result.importances_mean
perm_stds = perm_result.importances_std

print("=== Permutation importance (validation set) stats ===")
print("Shape of importances:", perm_result.importances.shape)
print("Min perm_mean:", perm_means.min())
print("Max perm_mean:", perm_means.max())
print("Number of exactly-zero perm_mean:", np.sum(perm_means == 0.0))
print()

perm_imp_df = (
    pd.DataFrame({
        "feature": feat_names,
        "perm_importance_mean": perm_means,
        "perm_importance_std": perm_stds,
    })
    .sort_values("perm_importance_mean", ascending=False)
)

print("Top-10 features by permutation importance (mean):\n",
      perm_imp_df.head(10))
print("\nBottom-10 features by permutation importance (mean):\n",
      perm_imp_df.tail(10))
print()

top20_perm = set(perm_imp_df["feature"].head(20))
top40_perm = set(perm_imp_df["feature"].head(40))

In [ ]:
# 2.4 Merge three views into summary_df
corr_df = corr_sorted.reset_index()
corr_df.columns = ["feature", "pearson_corr_with_Tc"]

summary_df = (
    rf_imp_df
    .merge(perm_imp_df, on="feature", how="outer")
    .merge(corr_df, on="feature", how="outer")
)

print("=== Summary table sanity check ===")
print("Summary_df shape:", summary_df.shape)
print("Any NaNs in rf_importance?:", summary_df["rf_importance"].isna().any())
print("Any NaNs in perm_importance_mean?:", summary_df["perm_importance_mean"].isna().any())
print("Any NaNs in pearson_corr_with_Tc?:", summary_df["pearson_corr_with_Tc"].isna().any())
print()

In [ ]:
# 2.5 Shortlist = intersection of Top-40 from all three methods
shortlist_features = sorted(list(top40_corr & top40_rf & top40_perm))
print(f"Shortlist size (intersection of Top-40): {len(shortlist_features)}")
print("Shortlist features:\n", shortlist_features)
print()

shortlist_df = summary_df[summary_df["feature"].isin(shortlist_features)].copy()
shortlist_df["description"] = shortlist_df["feature"].map(describe_feature)

cols_order = [
    "feature",
    "description",
    "rf_importance",
    "perm_importance_mean",
    "perm_importance_std",
    "pearson_corr_with_Tc",
]
shortlist_df = shortlist_df[cols_order]

shortlist_path = "important_features_shortlist_superconductivity.csv"
shortlist_df.to_csv(shortlist_path, index=False)
print(f"Saved shortlist CSV to: {shortlist_path}")

In [ ]:
# 2.6 Longlist = union of Top-20 from all three methods
longlist_features = sorted(list(top20_corr | top20_rf | top20_perm))
print(f"\nLonglist size (union of Top-20): {len(longlist_features)}")
print("Longlist features:\n", longlist_features)
print()

longlist_df = summary_df[summary_df["feature"].isin(longlist_features)].copy()
longlist_df["description"] = longlist_df["feature"].map(describe_feature)
longlist_df = longlist_df[cols_order]

longlist_path = "important_features_longlist_superconductivity.csv"
longlist_df.to_csv(longlist_path, index=False)
print(f"Saved longlist CSV to: {longlist_path}")

In [ ]:
# ============================================================
# 3. XGBoost feature importance (Hamidieh-style)
#    Train on TRAIN, evaluate on VAL; verification is untouched.
# ============================================================
from xgboost import XGBRegressor
print("\n=== XGBoost (Hamidieh-style) on Train/Val split ===")

xgb = XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)

# Evaluate on validation set
y_val_pred_xgb = xgb.predict(X_val)
mse_xgb = mean_squared_error(y_val, y_val_pred_xgb)
rmse_xgb = mse_xgb ** 0.5
r2_xgb = r2_score(y_val, y_val_pred_xgb)

print(f"XGBoost Val MSE:  {mse_xgb:.4f}")
print(f"XGBoost Val RMSE: {rmse_xgb:.4f} K")
print(f"XGBoost Val R^2:  {r2_xgb:.4f}")

# Extract gain-based feature importance
booster = xgb.get_booster()
gain_importance = booster.get_score(importance_type="gain")

print("Raw keys from booster.get_score():", list(gain_importance.keys())[:10])

imp_xgb = pd.DataFrame(
    list(gain_importance.items()),
    columns=["feature", "gain_importance"]
)

# Ensure all features appear (unused features get 0)
imp_xgb = (
    imp_xgb
    .set_index("feature")
    .reindex(feat_names, fill_value=0.0)
    .reset_index()
)

imp_xgb = imp_xgb.sort_values("gain_importance", ascending=False).reset_index(drop=True)

print("\nTop 20 features by XGBoost gain importance:\n")
print(imp_xgb.head(20))

ham_output_path = "hamidieh_style_xgboost_feature_importance.csv"
imp_xgb.to_csv(ham_output_path, index=False)
print(f"\nSaved XGBoost feature importance to: {ham_output_path}")

In [ ]:
import pandas as pd

# ------------------------------------------------------------
# 1. Load the three CSV files
# ------------------------------------------------------------

shortlist_path = "important_features_shortlist_superconductivity.csv"
longlist_path = "important_features_longlist_superconductivity.csv"
ham_path = "hamidieh_style_xgboost_feature_importance.csv"

shortlist_df = pd.read_csv(shortlist_path)
longlist_df = pd.read_csv(longlist_path)
ham_df = pd.read_csv(ham_path)

print("=== Column names check ===")
print("Shortlist columns:", shortlist_df.columns.tolist())
print("Longlist  columns:", longlist_df.columns.tolist())
print("Hamidieh-style columns:", ham_df.columns.tolist())
print()

# ------------------------------------------------------------
# 2. Robustly determine the 'feature' column name
# ------------------------------------------------------------

def get_feature_col_name(df, preferred="feature"):
    """Return the name of the feature column in df.
    If 'preferred' exists, use it; otherwise use the first column."""
    cols = df.columns.tolist()
    if preferred in cols:
        return preferred
    else:
        print(f"WARNING: '{preferred}' not found in columns {cols}. "
              f"Using first column '{cols[0]}' as feature column.")
        return cols[0]

shortlist_feat_col = get_feature_col_name(shortlist_df, preferred="feature")
longlist_feat_col = get_feature_col_name(longlist_df, preferred="feature")
ham_feat_col = get_feature_col_name(ham_df, preferred="feature")

# ------------------------------------------------------------
# 3. Build feature sets
# ------------------------------------------------------------

shortlist_features = set(shortlist_df[shortlist_feat_col].tolist())
longlist_features = set(longlist_df[longlist_feat_col].tolist())

# For Hamidieh-style table, we also need the importance column
# to sort by. Try 'gain_importance' first, otherwise use the
# second column as a fallback.
if "gain_importance" in ham_df.columns:
    imp_col = "gain_importance"
else:
    # Fallback: use second column as importance
    if len(ham_df.columns) >= 2:
        imp_col = ham_df.columns[1]
        print(f"WARNING: 'gain_importance' not found. Using column '{imp_col}' "
              "as importance measure.")
    else:
        raise ValueError("Hamidieh-style CSV does not have enough columns.")

ham_df_sorted = ham_df.sort_values(imp_col, ascending=False)

# Take Top-16 and Top-37 features from Hamidieh-style ranking
top16_ham_features = set(ham_df_sorted[ham_feat_col].head(16).tolist())
top37_ham_features = set(ham_df_sorted[ham_feat_col].head(37).tolist())

print("=== Size summary ===")
print("Number of features in shortlist:", len(shortlist_features))
print("Number of features in longlist:", len(longlist_features))
print("Number of features in Hamidieh-style Top-16:", len(top16_ham_features))
print("Number of features in Hamidieh-style Top-37:", len(top37_ham_features))
print()

# ------------------------------------------------------------
# 4. Intersection: shortlist ∩ Hamidieh-style Top-16
# ------------------------------------------------------------

shortlist_ham16_intersection = shortlist_features & top16_ham_features

print("Features that appear in BOTH shortlist and Hamidieh-style Top-16:")
print(sorted(list(shortlist_ham16_intersection)))
print("Count:", len(shortlist_ham16_intersection))
print()

# ------------------------------------------------------------
# 5. Overlap rate between longlist and Hamidieh-style Top-37
# ------------------------------------------------------------

longlist_ham37_intersection = longlist_features & top37_ham_features

intersect_count = len(longlist_ham37_intersection)
top37_count = len(top37_ham_features)
longlist_count = len(longlist_features)

overlap_rate_vs_top37 = intersect_count / top37_count if top37_count > 0 else 0.0
overlap_rate_vs_longlist = intersect_count / longlist_count if longlist_count > 0 else 0.0

print("Intersection between longlist and Hamidieh-style Top-37:")
print(sorted(list(longlist_ham37_intersection)))
print("Intersection count:", intersect_count)
print(f"Overlap rate vs Hamidieh-style Top-37: {overlap_rate_vs_top37:.3f}")
print(f"Overlap rate vs longlist: {overlap_rate_vs_longlist:.3f}")


In [ ]:
# ============================================================
# 4. Simple Neural Network for Temperature Prediction
#    Using PyTorch for a feedforward neural network
#    Uses the existing train/validation/verification split (70/15/15)
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Use the existing train/validation/verification split from earlier cells
print("=== Using existing train/validation/verification split ===")
print(f"Train size:          {X_train.shape[0]}")
print(f"Validation size:     {X_val.shape[0]}")
print(f"Verification size:   {X_verif.shape[0]}")
print()

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print()

# Standardize features (important for neural networks)
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled = scaler_X.transform(X_val)
X_verif_scaled = scaler_X.transform(X_verif)

# Reshape y for scaler (needs 2D array)
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
y_val_scaled = scaler_y.transform(y_val.values.reshape(-1, 1)).flatten()
y_verif_scaled = scaler_y.transform(y_verif.values.reshape(-1, 1)).flatten()

print("=== Data standardization complete ===")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_train_scaled mean (should be ~0): {X_train_scaled.mean():.6f}")
print(f"X_train_scaled std (should be ~1): {X_train_scaled.std():.6f}")
print()

# Create PyTorch Dataset
class SuperconductorDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = SuperconductorDataset(X_train_scaled, y_train_scaled)
val_dataset = SuperconductorDataset(X_val_scaled, y_val_scaled)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# Define simple neural network
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_sizes=[128, 64, 32], dropout_rate=0.2):
        super(SimpleNN, self).__init__()
        
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_size = hidden_size
        
        # Output layer (single value for temperature)
        layers.append(nn.Linear(prev_size, 1))
        
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x).squeeze()

# Initialize model
input_size = X_train_scaled.shape[1]
model = SimpleNN(input_size, hidden_sizes=[128, 64, 32], dropout_rate=0.2).to(device)

print("=== Neural Network Architecture ===")
print(model)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print()

# Loss function and optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

# Helper function to evaluate model and compute metrics in original scale
def evaluate_model_metrics(model, X_scaled, y_true, scaler_y, device):
    """Evaluate model and return metrics in original scale"""
    model.eval()
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X_scaled).to(device)
        y_pred_scaled = model(X_tensor).cpu().numpy()
    
    # Inverse transform to original scale
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    
    # Calculate metrics
    mse = mean_squared_error(y_true, y_pred)
    rmse = mse ** 0.5
    r2 = r2_score(y_true, y_pred)
    
    # Also compute scaled loss (for consistency with training)
    y_true_scaled = scaler_y.transform(y_true.values.reshape(-1, 1)).flatten()
    mse_scaled = mean_squared_error(y_true_scaled, y_pred_scaled)
    
    return {
        'mse': mse,
        'rmse': rmse,
        'r2': r2,
        'mse_scaled': mse_scaled
    }

# Training loop with comprehensive metric tracking
num_epochs = 100
metrics_history = {
    'epoch': [],
    'train_loss': [],
    'train_mse': [],
    'train_rmse': [],
    'train_r2': [],
    'val_loss': [],
    'val_mse': [],
    'val_rmse': [],
    'val_r2': [],
    'verif_loss': [],
    'verif_mse': [],
    'verif_rmse': [],
    'verif_r2': []
}

best_val_loss = float('inf')
patience = 20
patience_counter = 0

# Create verification dataset loader
verif_dataset = SuperconductorDataset(X_verif_scaled, y_verif_scaled)
verif_loader = DataLoader(verif_dataset, batch_size=64, shuffle=False)

print("=== Training Neural Network ===")
for epoch in range(num_epochs):
    # Training phase
    model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    
    # Evaluate on all sets (in original scale for RMSE and R²)
    train_metrics = evaluate_model_metrics(model, X_train_scaled, y_train, scaler_y, device)
    val_metrics = evaluate_model_metrics(model, X_val_scaled, y_val, scaler_y, device)
    verif_metrics = evaluate_model_metrics(model, X_verif_scaled, y_verif, scaler_y, device)
    
    # Store metrics
    metrics_history['epoch'].append(epoch + 1)
    metrics_history['train_loss'].append(train_metrics['mse_scaled'])
    metrics_history['train_mse'].append(train_metrics['mse'])
    metrics_history['train_rmse'].append(train_metrics['rmse'])
    metrics_history['train_r2'].append(train_metrics['r2'])
    metrics_history['val_loss'].append(val_metrics['mse_scaled'])
    metrics_history['val_mse'].append(val_metrics['mse'])
    metrics_history['val_rmse'].append(val_metrics['rmse'])
    metrics_history['val_r2'].append(val_metrics['r2'])
    metrics_history['verif_loss'].append(verif_metrics['mse_scaled'])
    metrics_history['verif_mse'].append(verif_metrics['mse'])
    metrics_history['verif_rmse'].append(verif_metrics['rmse'])
    metrics_history['verif_r2'].append(verif_metrics['r2'])
    
    val_loss = val_metrics['mse_scaled']
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Save best model
        torch.save(model.state_dict(), 'best_nn_model.pth')
    else:
        patience_counter += 1
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}]")
        print(f"  Train - Loss: {train_metrics['mse_scaled']:.6f}, RMSE: {train_metrics['rmse']:.4f} K, R²: {train_metrics['r2']:.4f}")
        print(f"  Val   - Loss: {val_metrics['mse_scaled']:.6f}, RMSE: {val_metrics['rmse']:.4f} K, R²: {val_metrics['r2']:.4f}")
        print(f"  Verif - Loss: {verif_metrics['mse_scaled']:.6f}, RMSE: {verif_metrics['rmse']:.4f} K, R²: {verif_metrics['r2']:.4f}")
    
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

# Load best model
model.load_state_dict(torch.load('best_nn_model.pth'))

# Save metrics to CSV
metrics_df = pd.DataFrame(metrics_history)
csv_filename = 'nn_training_metrics.csv'
metrics_df.to_csv(csv_filename, index=False)
print(f"=== Saved training metrics to {csv_filename} ===")
print(f"Metrics tracked for {len(metrics_history['epoch'])} epochs")
print()

# Create comprehensive plots (Verification set only)
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Plot 1: Loss curves (Verification only)
axes[0, 0].plot(metrics_history['epoch'], metrics_history['verif_loss'], label='Verification Loss', linewidth=2, color='green')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('MSE Loss (scaled)')
axes[0, 0].set_title('Neural Network: Verification Loss Curve')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: RMSE curves (Verification only)
axes[0, 1].plot(metrics_history['epoch'], metrics_history['verif_rmse'], label='Verification RMSE', linewidth=2, color='blue')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('RMSE (K)')
axes[0, 1].set_title('Neural Network: Verification RMSE Curve')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: R² curves (Verification only)
axes[1, 0].plot(metrics_history['epoch'], metrics_history['verif_r2'], label='Verification R²', linewidth=2, color='red')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('R² Score')
axes[1, 0].set_title('Neural Network: Verification R² Score Curve')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Combined view (RMSE and R²) - Verification only
ax2 = axes[1, 1]
ax2_twin = ax2.twinx()
line1 = ax2.plot(metrics_history['epoch'], metrics_history['verif_rmse'], 'b-', label='Verif RMSE', linewidth=2)
line2 = ax2_twin.plot(metrics_history['epoch'], metrics_history['verif_r2'], 'r-', label='Verif R²', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('RMSE (K)', color='b')
ax2_twin.set_ylabel('R² Score', color='r')
ax2.set_title('Neural Network: Verification Metrics (RMSE & R²)')
ax2.tick_params(axis='y', labelcolor='b')
ax2_twin.tick_params(axis='y', labelcolor='r')
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax2.legend(lines, labels, loc='center right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plot_filename = 'nn_training_curves.png'
plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
print(f"=== Saved training curves to {plot_filename} ===")
plt.show()

# Evaluate on validation set (in original scale)
model.eval()
with torch.no_grad():
    X_val_tensor = torch.FloatTensor(X_val_scaled).to(device)
    y_val_pred_scaled = model(X_val_tensor).cpu().numpy()

# Inverse transform predictions
y_val_pred = scaler_y.inverse_transform(y_val_pred_scaled.reshape(-1, 1)).flatten()

# Calculate metrics
mse_nn = mean_squared_error(y_val, y_val_pred)
rmse_nn = mse_nn ** 0.5
r2_nn = r2_score(y_val, y_val_pred)

print("\n=== Neural Network Performance on Validation Set ===")
print(f"Val MSE:  {mse_nn:.4f}")
print(f"Val RMSE: {rmse_nn:.4f} K")
print(f"Val R^2:  {r2_nn:.4f}")
print()

# Compare with previous models (if they exist)
try:
    print("=== Model Comparison (Validation Set) ===")
    if 'rmse_rf' in globals() and 'r2_rf' in globals():
        print(f"Random Forest - RMSE: {rmse_rf:.4f} K, R²: {r2_rf:.4f}")
    if 'rmse_xgb' in globals() and 'r2_xgb' in globals():
        print(f"XGBoost      - RMSE: {rmse_xgb:.4f} K, R²: {r2_xgb:.4f}")
    print(f"Neural Net   - RMSE: {rmse_nn:.4f} K, R²: {r2_nn:.4f}")
    print()
except:
    print("=== Neural Network Results (Validation Set) ===")
    print(f"RMSE: {rmse_nn:.4f} K")
    print(f"R²: {r2_nn:.4f}")
    print()

# Evaluate on verification set (in original scale)
model.eval()
with torch.no_grad():
    X_verif_tensor = torch.FloatTensor(X_verif_scaled).to(device)
    y_verif_pred_scaled = model(X_verif_tensor).cpu().numpy()

# Inverse transform predictions
y_verif_pred = scaler_y.inverse_transform(y_verif_pred_scaled.reshape(-1, 1)).flatten()

# Calculate metrics
mse_verif = mean_squared_error(y_verif, y_verif_pred)
rmse_verif = mse_verif ** 0.5
r2_verif = r2_score(y_verif, y_verif_pred)

print("=== Neural Network Performance on Verification Set ===")
print(f"Verif MSE:  {mse_verif:.4f}")
print(f"Verif RMSE: {rmse_verif:.4f} K")
print(f"Verif R^2:  {r2_verif:.4f}")
print()

# Create summary entry for comparison table
config_id = "baseline"  # Change this for different configurations
best_epoch_idx = np.argmin(metrics_history['val_loss'])
best_epoch = metrics_history['epoch'][best_epoch_idx]

nn_summary = {
    'model_name': 'Neural Network',
    'config_id': config_id,
    'train_mse': metrics_history['train_mse'][best_epoch_idx],
    'train_rmse': metrics_history['train_rmse'][best_epoch_idx],
    'train_r2': metrics_history['train_r2'][best_epoch_idx],
    'val_mse': metrics_history['val_mse'][best_epoch_idx],
    'val_rmse': metrics_history['val_rmse'][best_epoch_idx],
    'val_r2': metrics_history['val_r2'][best_epoch_idx],
    'verif_mse': metrics_history['verif_mse'][best_epoch_idx],
    'verif_rmse': metrics_history['verif_rmse'][best_epoch_idx],
    'verif_r2': metrics_history['verif_r2'][best_epoch_idx],
    'best_epoch': best_epoch,
    'hidden_layers': '[128, 64, 32]',
    'dropout': 0.2,
    'learning_rate': 0.001,
    'batch_size': 64
}

nn_summary_df = pd.DataFrame([nn_summary])
summary_filename = f'nn_config_{config_id}_summary.csv'
nn_summary_df.to_csv(summary_filename, index=False)
print(f"=== Saved model summary to {summary_filename} ===")
print("\nBest epoch summary:")
print(nn_summary_df.to_string(index=False))


In [ ]:
# ============================================================
# MSE Curve for Training Epochs (Original Scale)
# ============================================================
# Check if metrics_history exists and has MSE data
if 'metrics_history' not in globals() or len(metrics_history.get('epoch', [])) == 0:
    print("ERROR: metrics_history not found or empty.")
    print("Please run the neural network training cell (Cell 11) first to generate the metrics.")
elif 'train_mse' not in metrics_history:
    print("ERROR: MSE metrics not found in metrics_history.")
    print("Please re-run the neural network training cell (Cell 11) to track MSE values.")
    print("The training cell should track train_mse, val_mse, and verif_mse.")
else:
    # Create dedicated MSE curve plot showing MSE in original scale (K²)
    plt.figure(figsize=(12, 6))
    plt.plot(metrics_history['epoch'], metrics_history['train_mse'], 
             label='Train MSE', linewidth=2.5, marker='o', markersize=4, markevery=5)
    plt.plot(metrics_history['epoch'], metrics_history['val_mse'], 
             label='Validation MSE', linewidth=2.5, marker='s', markersize=4, markevery=5)
    plt.plot(metrics_history['epoch'], metrics_history['verif_mse'], 
             label='Verification MSE', linewidth=2.5, marker='^', markersize=4, markevery=5)
    plt.xlabel('Epoch', fontsize=12, fontweight='bold')
    plt.ylabel('MSE (K²)', fontsize=12, fontweight='bold')
    plt.title('Neural Network: Mean Squared Error (MSE) Curves', fontsize=14, fontweight='bold')
    plt.legend(fontsize=11, loc='best')
    plt.grid(True, alpha=0.3, linestyle='--')
    plt.tight_layout()

    # Save the plot
    mse_plot_filename = 'nn_mse_curves.png'
    plt.savefig(mse_plot_filename, dpi=300, bbox_inches='tight')
    print(f"=== Saved MSE curves to {mse_plot_filename} ===")
    plt.show()

    # Print final MSE values
    print("\n=== Final MSE Values (Original Scale) ===")
    final_epoch = len(metrics_history['epoch']) - 1
    print(f"Train MSE:     {metrics_history['train_mse'][final_epoch]:.4f} K²")
    print(f"Validation MSE: {metrics_history['val_mse'][final_epoch]:.4f} K²")
    print(f"Verification MSE: {metrics_history['verif_mse'][final_epoch]:.4f} K²")
    print()


In [ ]:
# Modified version based on Ziteng's SimpleNN
# to compare different NN structures
# ============================================================
# 4. Simple Neural Network for Temperature Prediction
#    Using PyTorch for a feedforward neural network
#    Uses the existing train/validation/verification split (70/15/15)
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Use the existing train/validation/verification split from earlier cells
print("=== Using existing train/validation/verification split ===")
print(f"Train size:          {X_train.shape[0]}")
print(f"Validation size:     {X_val.shape[0]}")
print(f"Verification size:   {X_verif.shape[0]}")
print()

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print()

# Standardize features (important for neural networks)
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled = scaler_X.transform(X_val)
X_verif_scaled = scaler_X.transform(X_verif)

# Reshape y for scaler (needs 2D array)
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
y_val_scaled = scaler_y.transform(y_val.values.reshape(-1, 1)).flatten()
y_verif_scaled = scaler_y.transform(y_verif.values.reshape(-1, 1)).flatten()

print("=== Data standardization complete ===")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_train_scaled mean (should be ~0): {X_train_scaled.mean():.6f}")
print(f"X_train_scaled std (should be ~1): {X_train_scaled.std():.6f}")
print()

# Create PyTorch Dataset
class SuperconductorDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = SuperconductorDataset(X_train_scaled, y_train_scaled)
val_dataset = SuperconductorDataset(X_val_scaled, y_val_scaled)

# NOTE: loaders will be created inside the loop (batch_size can vary per config)
# train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# Define simple neural network
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_sizes=[128, 64, 32], dropout_rate=0.2):
        super(SimpleNN, self).__init__()
        
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_size = hidden_size
        
        # Output layer (single value for temperature)
        layers.append(nn.Linear(prev_size, 1))
        
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x).squeeze()

# Helper function to evaluate model and compute metrics in original scale
def evaluate_model_metrics(model, X_scaled, y_true, scaler_y, device):
    """Evaluate model and return metrics in original scale"""
    model.eval()
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X_scaled).to(device)
        y_pred_scaled = model(X_tensor).cpu().numpy()
    
    # Inverse transform to original scale
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    
    # Calculate metrics
    mse = mean_squared_error(y_true, y_pred)
    rmse = mse ** 0.5
    r2 = r2_score(y_true, y_pred)
    
    # Also compute scaled loss (for consistency with training)
    y_true_scaled = scaler_y.transform(y_true.values.reshape(-1, 1)).flatten()
    mse_scaled = mean_squared_error(y_true_scaled, y_pred_scaled)
    
    return {
        'mse': mse,
        'rmse': rmse,
        'r2': r2,
        'mse_scaled': mse_scaled
    }

# Create verification dataset loader
verif_dataset = SuperconductorDataset(X_verif_scaled, y_verif_scaled)
# verif_loader = DataLoader(verif_dataset, batch_size=64, shuffle=False)
# (verif_loader is not used in the current code; we evaluate on full arrays)

# ============================================================
# Experiment configurations (configs to loop over)
# ============================================================
experiment_configs = [
    {
        "config_id": "baseline",
        "hidden_sizes": [128, 64, 32],
        "dropout": 0.2,
        "learning_rate": 1e-3,
        "batch_size": 64,
        "weight_decay": 1e-5,
    },
    {
        "config_id": "no_dropout",
        "hidden_sizes": [128, 64, 32],
        "dropout": 0.0,
        "learning_rate": 1e-3,
        "batch_size": 64,
        "weight_decay": 1e-5,
    },
    {
        "config_id": "wider",
        "hidden_sizes": [256, 128, 64],
        "dropout": 0.2,
        "learning_rate": 1e-3,
        "batch_size": 64,
        "weight_decay": 1e-5,
    },
    {
        "config_id": "small_lr",
        "hidden_sizes": [128, 64, 32],
        "dropout": 0.2,
        "learning_rate": 5e-4,
        "batch_size": 64,
        "weight_decay": 1e-5,
    },
    {
        "config_id": "small_net",
        "hidden_sizes": [64, 32],
        "dropout": 0.2,
        "learning_rate": 1e-3,
        "batch_size": 32,
        "weight_decay": 0.0,
    },
]

all_summaries = []

# Initialize model (and loop over configs)
input_size = X_train_scaled.shape[1]

for cfg in experiment_configs:
    config_id     = cfg["config_id"]
    hidden_sizes  = cfg["hidden_sizes"]
    dropout_rate  = cfg["dropout"]
    learning_rate = cfg["learning_rate"]
    batch_size    = cfg["batch_size"]
    weight_decay  = cfg["weight_decay"]

    print("\n" + "=" * 80)
    print(f"=== Training config: {config_id} ===")
    print(f"hidden_sizes={hidden_sizes}, dropout={dropout_rate}, "
          f"lr={learning_rate}, batch_size={batch_size}, weight_decay={weight_decay}")
    print("=" * 80 + "\n")

    # (re)create loaders for this config
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
    verif_loader = DataLoader(verif_dataset, batch_size=batch_size, shuffle=False)

    model = SimpleNN(input_size, hidden_sizes=hidden_sizes, dropout_rate=dropout_rate).to(device)

    print("=== Neural Network Architecture ===")
    print(model)
    print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
    print()

    # Loss function and optimizer
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

    # Training loop with comprehensive metric tracking
    num_epochs = 100
    metrics_history = {
        'epoch': [],
        'train_loss': [],
        'train_mse': [],
        'train_rmse': [],
        'train_r2': [],
        'val_loss': [],
        'val_mse': [],
        'val_rmse': [],
        'val_r2': [],
        'verif_loss': [],
        'verif_mse': [],
        'verif_rmse': [],
        'verif_r2': []
    }

    best_val_loss = float('inf')
    patience = 20
    patience_counter = 0

    print("=== Training Neural Network ===")
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        
        # Evaluate on all sets (in original scale for RMSE and R²)
        train_metrics = evaluate_model_metrics(model, X_train_scaled, y_train, scaler_y, device)
        val_metrics   = evaluate_model_metrics(model, X_val_scaled,   y_val,   scaler_y, device)
        verif_metrics = evaluate_model_metrics(model, X_verif_scaled, y_verif, scaler_y, device)
        
        # Store metrics
        metrics_history['epoch'].append(epoch + 1)
        metrics_history['train_loss'].append(train_metrics['mse_scaled'])
        metrics_history['train_mse'].append(train_metrics['mse'])
        metrics_history['train_rmse'].append(train_metrics['rmse'])
        metrics_history['train_r2'].append(train_metrics['r2'])
        metrics_history['val_loss'].append(val_metrics['mse_scaled'])
        metrics_history['val_mse'].append(val_metrics['mse'])
        metrics_history['val_rmse'].append(val_metrics['rmse'])
        metrics_history['val_r2'].append(val_metrics['r2'])
        metrics_history['verif_loss'].append(verif_metrics['mse_scaled'])
        metrics_history['verif_mse'].append(verif_metrics['mse'])
        metrics_history['verif_rmse'].append(verif_metrics['rmse'])
        metrics_history['verif_r2'].append(verif_metrics['r2'])
        
        val_loss = val_metrics['mse_scaled']
        
        # Learning rate scheduling
        scheduler.step(val_loss)
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            # Save best model (per config)
            torch.save(model.state_dict(), f'best_nn_model_{config_id}.pth')
        else:
            patience_counter += 1
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}]")
            print(f"  Train - Loss: {train_metrics['mse_scaled']:.6f}, RMSE: {train_metrics['rmse']:.4f} K, R²: {train_metrics['r2']:.4f}")
            print(f"  Val   - Loss: {val_metrics['mse_scaled']:.6f}, RMSE: {val_metrics['rmse']:.4f} K, R²: {val_metrics['r2']:.4f}")
            print(f"  Verif - Loss: {verif_metrics['mse_scaled']:.6f}, RMSE: {verif_metrics['rmse']:.4f} K, R²: {verif_metrics['r2']:.4f}")
        
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1} (config: {config_id})")
            break

    # Load best model
    model.load_state_dict(torch.load(f'best_nn_model_{config_id}.pth'))

    # Save metrics to CSV
    metrics_df = pd.DataFrame(metrics_history)
    csv_filename = f'nn_training_metrics_{config_id}.csv'
    metrics_df.to_csv(csv_filename, index=False)
    print(f"=== Saved training metrics to {csv_filename} ===")
    print(f"Metrics tracked for {len(metrics_history['epoch'])} epochs")
    print()

    # Create comprehensive plots (Verification set only)
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    # Plot 1: Loss curves (Verification only)
    axes[0, 0].plot(metrics_history['epoch'], metrics_history['verif_loss'], label='Verification Loss', linewidth=2, color='green')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('MSE Loss (scaled)')
    axes[0, 0].set_title(f'Neural Network: Verification Loss Curve ({config_id})')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Plot 2: RMSE curves (Verification only)
    axes[0, 1].plot(metrics_history['epoch'], metrics_history['verif_rmse'], label='Verification RMSE', linewidth=2, color='blue')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('RMSE (K)')
    axes[0, 1].set_title(f'Neural Network: Verification RMSE Curve ({config_id})')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # Plot 3: R² curves (Verification only)
    axes[1, 0].plot(metrics_history['epoch'], metrics_history['verif_r2'], label='Verification R²', linewidth=2, color='red')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('R² Score')
    axes[1, 0].set_title(f'Neural Network: Verification R² Score Curve ({config_id})')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Plot 4: Combined view (RMSE and R²) - Verification only
    ax2 = axes[1, 1]
    ax2_twin = ax2.twinx()
    line1 = ax2.plot(metrics_history['epoch'], metrics_history['verif_rmse'], 'b-', label='Verif RMSE', linewidth=2)
    line2 = ax2_twin.plot(metrics_history['epoch'], metrics_history['verif_r2'], 'r-', label='Verif R²', linewidth=2)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('RMSE (K)', color='b')
    ax2_twin.set_ylabel('R² Score', color='r')
    ax2.set_title(f'Neural Network: Verification Metrics (RMSE & R²) ({config_id})')
    ax2.tick_params(axis='y', labelcolor='b')
    ax2_twin.tick_params(axis='y', labelcolor='r')
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax2.legend(lines, labels, loc='center right')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plot_filename = f'nn_training_curves_{config_id}.png'
    plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
    print(f"=== Saved training curves to {plot_filename} ===")
    plt.show()

    # Evaluate on validation set (in original scale)
    model.eval()
    with torch.no_grad():
        X_val_tensor = torch.FloatTensor(X_val_scaled).to(device)
        y_val_pred_scaled = model(X_val_tensor).cpu().numpy()

    # Inverse transform predictions
    y_val_pred = scaler_y.inverse_transform(y_val_pred_scaled.reshape(-1, 1)).flatten()

    # Calculate metrics
    mse_nn = mean_squared_error(y_val, y_val_pred)
    rmse_nn = mse_nn ** 0.5
    r2_nn = r2_score(y_val, y_val_pred)

    print("\n=== Neural Network Performance on Validation Set ===")
    print(f"(config: {config_id})")
    print(f"Val MSE:  {mse_nn:.4f}")
    print(f"Val RMSE: {rmse_nn:.4f} K")
    print(f"Val R^2:  {r2_nn:.4f}")
    print()

    # Compare with previous models (if they exist)
    try:
        print("=== Model Comparison (Validation Set) ===")
        if 'rmse_rf' in globals() and 'r2_rf' in globals():
            print(f"Random Forest - RMSE: {rmse_rf:.4f} K, R²: {r2_rf:.4f}")
        if 'rmse_xgb' in globals() and 'r2_xgb' in globals():
            print(f"XGBoost      - RMSE: {rmse_xgb:.4f} K, R²: {r2_xgb:.4f}")
        print(f"Neural Net   - RMSE: {rmse_nn:.4f} K, R²: {r2_nn:.4f}")
        print()
    except:
        print("=== Neural Network Results (Validation Set) ===")
        print(f"RMSE: {rmse_nn:.4f} K")
        print(f"R²: {r2_nn:.4f}")
        print()

    # Evaluate on verification set (in original scale)
    model.eval()
    with torch.no_grad():
        X_verif_tensor = torch.FloatTensor(X_verif_scaled).to(device)
        y_verif_pred_scaled = model(X_verif_tensor).cpu().numpy()

    # Inverse transform predictions
    y_verif_pred = scaler_y.inverse_transform(y_verif_pred_scaled.reshape(-1, 1)).flatten()

    # Calculate metrics
    mse_verif = mean_squared_error(y_verif, y_verif_pred)
    rmse_verif = mse_verif ** 0.5
    r2_verif = r2_score(y_verif, y_verif_pred)

    print("=== Neural Network Performance on Verification Set ===")
    print(f"(config: {config_id})")
    print(f"Verif MSE:  {mse_verif:.4f}")
    print(f"Verif RMSE: {rmse_verif:.4f} K")
    print(f"Verif R^2:  {r2_verif:.4f}")
    print()

    # Create summary entry for comparison table
    best_epoch_idx = np.argmin(metrics_history['val_loss'])
    best_epoch = metrics_history['epoch'][best_epoch_idx]

    nn_summary = {
        'model_name': 'Neural Network',
        'config_id': config_id,
        'train_mse': metrics_history['train_mse'][best_epoch_idx],
        'train_rmse': metrics_history['train_rmse'][best_epoch_idx],
        'train_r2': metrics_history['train_r2'][best_epoch_idx],
        'val_mse': metrics_history['val_mse'][best_epoch_idx],
        'val_rmse': metrics_history['val_rmse'][best_epoch_idx],
        'val_r2': metrics_history['val_r2'][best_epoch_idx],
        'verif_mse': metrics_history['verif_mse'][best_epoch_idx],
        'verif_rmse': metrics_history['verif_rmse'][best_epoch_idx],
        'verif_r2': metrics_history['verif_r2'][best_epoch_idx],
        'best_epoch': best_epoch,
        'hidden_layers': str(hidden_sizes),
        'dropout': dropout_rate,
        'learning_rate': learning_rate,
        'batch_size': batch_size,
        'weight_decay': weight_decay,
    }

    nn_summary_df = pd.DataFrame([nn_summary])
    summary_filename = f'nn_config_{config_id}_summary.csv'
    nn_summary_df.to_csv(summary_filename, index=False)
    print(f"=== Saved model summary to {summary_filename} ===")
    print("\nBest epoch summary:")
    print(nn_summary_df.to_string(index=False))

    all_summaries.append(nn_summary)

# After all configs: combined comparison table
all_summaries_df = pd.DataFrame(all_summaries)
all_summaries_df.to_csv('nn_all_configs_summary.csv', index=False)
print("\n=== Saved all-configs summary to nn_all_configs_summary.csv ===")
print(all_summaries_df.to_string(index=False))


In [ ]:
# Modified version based on Ziteng's SimpleNN
# to compare standard GD & SGD
# ============================================================
# 4. Simple Neural Network for Temperature Prediction
#    Using PyTorch for a feedforward neural network
#    Uses the existing train/validation/verification split (70/15/15)
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Use the existing train/validation/verification split from earlier cells
print("=== Using existing train/validation/verification split ===")
# Assuming X_train, X_val, X_verif, y_train, y_val, y_verif exist in your environment
print(f"Train size:          {X_train.shape[0]}")
print(f"Validation size:     {X_val.shape[0]}")
print(f"Verification size:   {X_verif.shape[0]}")
print()

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print()

# Standardize features (important for neural networks)
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled = scaler_X.transform(X_val)
X_verif_scaled = scaler_X.transform(X_verif)

# Reshape y for scaler (needs 2D array)
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
y_val_scaled = scaler_y.transform(y_val.values.reshape(-1, 1)).flatten()
y_verif_scaled = scaler_y.transform(y_verif.values.reshape(-1, 1)).flatten()

print("=== Data standardization complete ===")
print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_train_scaled mean (should be ~0): {X_train_scaled.mean():.6f}")
print(f"X_train_scaled std (should be ~1): {X_train_scaled.std():.6f}")
print()

# Create PyTorch Dataset
class SuperconductorDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = SuperconductorDataset(X_train_scaled, y_train_scaled)
val_dataset = SuperconductorDataset(X_val_scaled, y_val_scaled)

# Define simple neural network
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_sizes=[128, 64, 32], dropout_rate=0.2):
        super(SimpleNN, self).__init__()
        
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_size = hidden_size
        
        # Output layer (single value for temperature)
        layers.append(nn.Linear(prev_size, 1))
        
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x).squeeze()

# Helper function to evaluate model and compute metrics in original scale
def evaluate_model_metrics(model, X_scaled, y_true, scaler_y, device):
    """Evaluate model and return metrics in original scale"""
    model.eval()
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X_scaled).to(device)
        y_pred_scaled = model(X_tensor).cpu().numpy()
    
    # Inverse transform to original scale
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    
    # Calculate metrics
    mse = mean_squared_error(y_true, y_pred)
    rmse = mse ** 0.5
    r2 = r2_score(y_true, y_pred)
    
    # Scaled loss (for consistency with training)
    y_true_scaled = scaler_y.transform(y_true.values.reshape(-1, 1)).flatten()
    mse_scaled = mean_squared_error(y_true_scaled, y_pred_scaled)
    
    return {
        'mse': mse,
        'rmse': rmse,
        'r2': r2,
        'mse_scaled': mse_scaled
    }

# Create verification dataset loader
verif_dataset = SuperconductorDataset(X_verif_scaled, y_verif_scaled)

# ============================================================
# MODIFIED: Experiment configurations for SGD vs Batch GD vs Adam
# ============================================================
experiment_configs = [
    {
        "config_id": "Standard_Batch_GD",
        "optimizer_type": "SGD",
        "batch_size": "FULL",        # Special flag for full dataset
        "learning_rate": 0.1,        # Batch GD is stable, can take larger steps
        "momentum": 0.0,             # Pure GD has no momentum
        "hidden_sizes": [128, 64, 32],
        "dropout": 0.2,
        "weight_decay": 0.0,
    },
    {
        "config_id": "Pure_SGD",
        "optimizer_type": "SGD",
        "batch_size": 1,             # 1 sample per step --> should be noisy
        "learning_rate": 0.0001,     # Should be very small for Pure SGD
        "momentum": 0.0,
        "hidden_sizes": [128, 64, 32],
        "dropout": 0.2,
        "weight_decay": 0.0,
    },
    {
        "config_id": "MiniBatch_SGD_Momentum", # This is the standard "SGD" in practice
        "optimizer_type": "SGD",
        "batch_size": 64,
        "learning_rate": 0.01,
        "momentum": 0.9,             # Momentum helps smooth out the noise
        "hidden_sizes": [128, 64, 32],
        "dropout": 0.2,
        "weight_decay": 1e-5,
    },
    {
        "config_id": "Adam_Baseline", # Original optimizer for comparison
        "optimizer_type": "Adam",
        "batch_size": 64,
        "learning_rate": 1e-3,
        "momentum": 0.0,             
        "hidden_sizes": [128, 64, 32],
        "dropout": 0.2,
        "weight_decay": 1e-5,
    }
]

all_summaries = []

# Initialize model (and loop over configs)
input_size = X_train_scaled.shape[1]

for cfg in experiment_configs:
    config_id     = cfg["config_id"]
    hidden_sizes  = cfg["hidden_sizes"]
    dropout_rate  = cfg["dropout"]
    learning_rate = cfg["learning_rate"]
    weight_decay  = cfg["weight_decay"]
    
    # OPTIMIZER SPECIFIC PARAMS
    optimizer_type = cfg["optimizer_type"]
    momentum       = cfg["momentum"]
    
    # BATCH SIZE LOGIC
    if cfg["batch_size"] == "FULL":
        batch_size = len(train_dataset)
        print(f"\n-> Note: Using FULL BATCH Gradient Descent (Batch Size: {batch_size})")
    else:
        batch_size = cfg["batch_size"]

    print("\n" + "=" * 80)
    print(f"=== Training config: {config_id} ===")
    print(f"Optimizer={optimizer_type}, BatchSize={batch_size}, LR={learning_rate}, Momentum={momentum}")
    print("=" * 80 + "\n")

    # (re)create loaders for this config
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=256, shuffle=False) # Large batch for faster eval
    verif_loader = DataLoader(verif_dataset, batch_size=256, shuffle=False)

    model = SimpleNN(input_size, hidden_sizes=hidden_sizes, dropout_rate=dropout_rate).to(device)

    print("=== Neural Network Architecture ===")
    print(model)
    print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
    print()

    # Loss function 
    criterion = nn.MSELoss()
    
    # SELECT OPTIMIZER BASED ON CONFIG
    if optimizer_type == "SGD":
        # PyTorch SGD supports momentum
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=momentum, weight_decay=weight_decay)
    elif optimizer_type == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

    # Training loop with comprehensive metric tracking
    num_epochs = 100
    metrics_history = {
        'epoch': [],
        'train_loss': [],
        'train_mse': [],
        'train_rmse': [],
        'train_r2': [],
        'val_loss': [],
        'val_mse': [],
        'val_rmse': [],
        'val_r2': [],
        'verif_loss': [],
        'verif_mse': [],
        'verif_rmse': [],
        'verif_r2': []
    }

    best_val_loss = float('inf')
    patience = 20
    patience_counter = 0

    print("=== Training Neural Network ===")
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        
        # Evaluate on all sets (in original scale for RMSE and R²)
        train_metrics = evaluate_model_metrics(model, X_train_scaled, y_train, scaler_y, device)
        val_metrics   = evaluate_model_metrics(model, X_val_scaled,   y_val,   scaler_y, device)
        verif_metrics = evaluate_model_metrics(model, X_verif_scaled, y_verif, scaler_y, device)
        
        # Store metrics
        metrics_history['epoch'].append(epoch + 1)
        metrics_history['train_loss'].append(train_metrics['mse_scaled'])
        metrics_history['train_mse'].append(train_metrics['mse'])
        metrics_history['train_rmse'].append(train_metrics['rmse'])
        metrics_history['train_r2'].append(train_metrics['r2'])
        metrics_history['val_loss'].append(val_metrics['mse_scaled'])
        metrics_history['val_mse'].append(val_metrics['mse'])
        metrics_history['val_rmse'].append(val_metrics['rmse'])
        metrics_history['val_r2'].append(val_metrics['r2'])
        metrics_history['verif_loss'].append(verif_metrics['mse_scaled'])
        metrics_history['verif_mse'].append(verif_metrics['mse'])
        metrics_history['verif_rmse'].append(verif_metrics['rmse'])
        metrics_history['verif_r2'].append(verif_metrics['r2'])
        
        val_loss = val_metrics['mse_scaled']
        
        # Learning rate scheduling
        scheduler.step(val_loss)
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            # Save best model (per config)
            torch.save(model.state_dict(), f'best_nn_model_{config_id}.pth')
        else:
            patience_counter += 1
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}]")
            print(f"  Train - Loss: {train_metrics['mse_scaled']:.6f}, RMSE: {train_metrics['rmse']:.4f} K, R²: {train_metrics['r2']:.4f}")
            print(f"  Val   - Loss: {val_metrics['mse_scaled']:.6f}, RMSE: {val_metrics['rmse']:.4f} K, R²: {val_metrics['r2']:.4f}")
            print(f"  Verif - Loss: {verif_metrics['mse_scaled']:.6f}, RMSE: {verif_metrics['rmse']:.4f} K, R²: {verif_metrics['r2']:.4f}")
        
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1} (config: {config_id})")
            break

    # Load best model
    model.load_state_dict(torch.load(f'best_nn_model_{config_id}.pth'))

    # Save metrics to CSV
    metrics_df = pd.DataFrame(metrics_history)
    csv_filename = f'nn_training_metrics_{config_id}.csv'
    metrics_df.to_csv(csv_filename, index=False)
    print(f"=== Saved training metrics to {csv_filename} ===")
    print(f"Metrics tracked for {len(metrics_history['epoch'])} epochs")
    print()

    # Create comprehensive plots (Verification set only)
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    # Plot 1: Loss curves (Verification only)
    axes[0, 0].plot(metrics_history['epoch'], metrics_history['verif_loss'], label='Verification Loss', linewidth=2, color='green')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('MSE Loss (scaled)')
    axes[0, 0].set_title(f'Neural Network: Verification Loss Curve ({config_id})')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Plot 2: RMSE curves (Verification only)
    axes[0, 1].plot(metrics_history['epoch'], metrics_history['verif_rmse'], label='Verification RMSE', linewidth=2, color='blue')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('RMSE (K)')
    axes[0, 1].set_title(f'Neural Network: Verification RMSE Curve ({config_id})')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # Plot 3: R² curves (Verification only)
    axes[1, 0].plot(metrics_history['epoch'], metrics_history['verif_r2'], label='Verification R²', linewidth=2, color='red')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('R² Score')
    axes[1, 0].set_title(f'Neural Network: Verification R² Score Curve ({config_id})')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Plot 4: Combined view (RMSE and R²) - Verification only
    ax2 = axes[1, 1]
    ax2_twin = ax2.twinx()
    line1 = ax2.plot(metrics_history['epoch'], metrics_history['verif_rmse'], 'b-', label='Verif RMSE', linewidth=2)
    line2 = ax2_twin.plot(metrics_history['epoch'], metrics_history['verif_r2'], 'r-', label='Verif R²', linewidth=2)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('RMSE (K)', color='b')
    ax2_twin.set_ylabel('R² Score', color='r')
    ax2.set_title(f'Neural Network: Verification Metrics (RMSE & R²) ({config_id})')
    ax2.tick_params(axis='y', labelcolor='b')
    ax2_twin.tick_params(axis='y', labelcolor='r')
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax2.legend(lines, labels, loc='center right')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plot_filename = f'nn_training_curves_{config_id}.png'
    plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
    print(f"=== Saved training curves to {plot_filename} ===")
    plt.show()

    # Evaluate on validation set (in original scale)
    model.eval()
    with torch.no_grad():
        X_val_tensor = torch.FloatTensor(X_val_scaled).to(device)
        y_val_pred_scaled = model(X_val_tensor).cpu().numpy()

    # Inverse transform predictions
    y_val_pred = scaler_y.inverse_transform(y_val_pred_scaled.reshape(-1, 1)).flatten()

    # Calculate metrics
    mse_nn = mean_squared_error(y_val, y_val_pred)
    rmse_nn = mse_nn ** 0.5
    r2_nn = r2_score(y_val, y_val_pred)

    print("\n=== Neural Network Performance on Validation Set ===")
    print(f"(config: {config_id})")
    print(f"Val MSE:  {mse_nn:.4f}")
    print(f"Val RMSE: {rmse_nn:.4f} K")
    print(f"Val R^2:  {r2_nn:.4f}")
    print()

    # Compare with previous models (if they exist)
    try:
        print("=== Model Comparison (Validation Set) ===")
        if 'rmse_rf' in globals() and 'r2_rf' in globals():
            print(f"Random Forest - RMSE: {rmse_rf:.4f} K, R²: {r2_rf:.4f}")
        if 'rmse_xgb' in globals() and 'r2_xgb' in globals():
            print(f"XGBoost      - RMSE: {rmse_xgb:.4f} K, R²: {r2_xgb:.4f}")
        print(f"Neural Net   - RMSE: {rmse_nn:.4f} K, R²: {r2_nn:.4f}")
        print()
    except:
        print("=== Neural Network Results (Validation Set) ===")
        print(f"RMSE: {rmse_nn:.4f} K")
        print(f"R²: {r2_nn:.4f}")
        print()

    # Evaluate on verification set (in original scale)
    model.eval()
    with torch.no_grad():
        X_verif_tensor = torch.FloatTensor(X_verif_scaled).to(device)
        y_verif_pred_scaled = model(X_verif_tensor).cpu().numpy()

    # Inverse transform predictions
    y_verif_pred = scaler_y.inverse_transform(y_verif_pred_scaled.reshape(-1, 1)).flatten()

    # Calculate metrics
    mse_verif = mean_squared_error(y_verif, y_verif_pred)
    rmse_verif = mse_verif ** 0.5
    r2_verif = r2_score(y_verif, y_verif_pred)

    print("=== Neural Network Performance on Verification Set ===")
    print(f"(config: {config_id})")
    print(f"Verif MSE:  {mse_verif:.4f}")
    print(f"Verif RMSE: {rmse_verif:.4f} K")
    print(f"Verif R^2:  {r2_verif:.4f}")
    print()

    # Create summary entry for comparison table
    best_epoch_idx = np.argmin(metrics_history['val_loss'])
    best_epoch = metrics_history['epoch'][best_epoch_idx]

    nn_summary = {
        'model_name': 'Neural Network',
        'config_id': config_id,
        'train_mse': metrics_history['train_mse'][best_epoch_idx],
        'train_rmse': metrics_history['train_rmse'][best_epoch_idx],
        'train_r2': metrics_history['train_r2'][best_epoch_idx],
        'val_mse': metrics_history['val_mse'][best_epoch_idx],
        'val_rmse': metrics_history['val_rmse'][best_epoch_idx],
        'val_r2': metrics_history['val_r2'][best_epoch_idx],
        'verif_mse': metrics_history['verif_mse'][best_epoch_idx],
        'verif_rmse': metrics_history['verif_rmse'][best_epoch_idx],
        'verif_r2': metrics_history['verif_r2'][best_epoch_idx],
        'best_epoch': best_epoch,
        'hidden_layers': str(hidden_sizes),
        'dropout': dropout_rate,
        'learning_rate': learning_rate,
        'batch_size': batch_size,
        'weight_decay': weight_decay,
        'optimizer': optimizer_type,
        'momentum': momentum
    }

    nn_summary_df = pd.DataFrame([nn_summary])
    summary_filename = f'nn_config_{config_id}_summary.csv'
    nn_summary_df.to_csv(summary_filename, index=False)
    print(f"=== Saved model summary to {summary_filename} ===")
    print("\nBest epoch summary:")
    print(nn_summary_df.to_string(index=False))

    all_summaries.append(nn_summary)

# After all configs: combined comparison table
all_summaries_df = pd.DataFrame(all_summaries)
all_summaries_df.to_csv('nn_all_configs_summary.csv', index=False)
print("\n=== Saved all-configs summary to nn_all_configs_summary.csv ===")
print(all_summaries_df.to_string(index=False))

# ============================================================
#  Final selection
#  - Evaluate all configs at once using all_summaries_df
#  - Rank them by a chosen metric (default: verification R²)
# ============================================================

# 1. Choose the selection metric
selection_metric = 'verif_r2'      # could also use 'val_r2', 'verif_rmse', etc.
higher_is_better = True           # R²: higher is better; RMSE/MSE: set this False

# 2. Rank all configurations
if higher_is_better:
    ranked_df = all_summaries_df.sort_values(by=selection_metric, ascending=False)
else:
    ranked_df = all_summaries_df.sort_values(by=selection_metric, ascending=True)

print("\n=== Ranking of all NN configs (by {}) ===".format(selection_metric))
print(
    ranked_df[
        [
            'config_id',
            'optimizer',
            'batch_size',
            'learning_rate',
            'momentum',
            'train_r2',
            'val_r2',
            'verif_r2',
            'train_rmse',
            'val_rmse',
            'verif_rmse',
            'best_epoch'
        ]
    ].to_string(index=False)
)

# 3. Take the best configuration
best_row = ranked_df.iloc[0]
best_config_id = best_row['config_id']
best_metric_value = best_row[selection_metric]

print("\n=== Best configuration selected ===")
print(f"Metric         : {selection_metric}")
print(f"Best config id : {best_config_id}")
print(f"{selection_metric} : {best_metric_value:.4f}")
print()


In [ ]:
# Modified version based on Ziteng's SimpleNN
# to compare different NN structures
# ============================================================
# 4. Simple Neural Network for Temperature Prediction
#    Using PyTorch for a feedforward neural network
#    Uses the existing train/validation/verification split (70/15/15)
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split  # <--- ADDED
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# helper to load feature names and make splits per dataset
# ------------------------------------------------------------

def load_feature_names(csv_path):
    """
    Load a 1D list of feature names from a CSV with a 'feature' column.
    """
    df_feat = pd.read_csv(csv_path)
    return df_feat['feature'].tolist()

def make_feature_split(feature_file, main_data_file='train.csv', target_col='critical_temp'):
    """
    Load main_data_file, select columns based on feature_file (or all features if None),
    then create a 70/15/15 split: train / val / verif.

    Returns:
        X_train, X_val, X_verif, y_train, y_val, y_verif, used_features
    """
    df = pd.read_csv(main_data_file)

    if feature_file is None:
        # Full feature set
        used_features = [c for c in df.columns if c != target_col]
        print(f"\n[Dataset] Using ALL {len(used_features)} features from {main_data_file}")
    else:
        feat_list = load_feature_names(feature_file)
        used_features = [f for f in feat_list if f in df.columns]
        print(f"\n[Dataset] Using {len(used_features)} features from {feature_file}")

    X = df[used_features].values
    y = df[target_col].values

    # 70/15/15 split (same rule)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.30, random_state=42
    )
    X_val, X_verif, y_val, y_verif = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=42
    )

    # Convert y to pandas Series so .values / .reshape work exactly as before
    y_train = pd.Series(y_train)
    y_val   = pd.Series(y_val)
    y_verif = pd.Series(y_verif)

    return X_train, X_val, X_verif, y_train, y_val, y_verif, used_features

# Create PyTorch Dataset
class SuperconductorDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Define simple neural network
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_sizes=[128, 64, 32], dropout_rate=0.2):
        super(SimpleNN, self).__init__()
        
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_size = hidden_size
        
        # Output layer (single value for temperature)
        layers.append(nn.Linear(prev_size, 1))
        
        self.model = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.model(x).squeeze()

# Helper function to evaluate model and compute metrics in original scale
def evaluate_model_metrics(model, X_scaled, y_true, scaler_y, device):
    """Evaluate model and return metrics in original scale"""
    model.eval()
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X_scaled).to(device)
        y_pred_scaled = model(X_tensor).cpu().numpy()
    
    # Inverse transform to original scale
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    
    # Calculate metrics
    mse = mean_squared_error(y_true, y_pred)
    rmse = mse ** 0.5
    r2 = r2_score(y_true, y_pred)
    
    # Also compute scaled loss (for consistency with training)
    y_true_scaled = scaler_y.transform(y_true.values.reshape(-1, 1)).flatten()
    mse_scaled = mean_squared_error(y_true_scaled, y_pred_scaled)
    
    return {
        'mse': mse,
        'rmse': rmse,
        'r2': r2,
        'mse_scaled': mse_scaled
    }

# ============================================================
# Experiment configurations (configs to loop over)
# ============================================================
experiment_configs = [
    {
        "config_id": "baseline",
        "hidden_sizes": [128, 64, 32],
        "dropout": 0.2,
        "learning_rate": 1e-3,
        "batch_size": 64,
        "weight_decay": 1e-5,
    },
    {
        "config_id": "no_dropout",
        "hidden_sizes": [128, 64, 32],
        "dropout": 0.0,
        "learning_rate": 1e-3,
        "batch_size": 64,
        "weight_decay": 1e-5,
    },
    {
        "config_id": "wider",
        "hidden_sizes": [256, 128, 64],
        "dropout": 0.2,
        "learning_rate": 1e-3,
        "batch_size": 64,
        "weight_decay": 1e-5,
    },
    {
        "config_id": "small_lr",
        "hidden_sizes": [128, 64, 32],
        "dropout": 0.2,
        "learning_rate": 5e-4,
        "batch_size": 64,
        "weight_decay": 1e-5,
    },
    {
        "config_id": "small_net",
        "hidden_sizes": [64, 32],
        "dropout": 0.2,
        "learning_rate": 1e-3,
        "batch_size": 32,
        "weight_decay": 0.0,
    },
]

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print()

# ============================================================
# Loop over 3 datasets (full, longlist, shortlist)
# ============================================================
dataset_specs = [
    ("full",     None),   # None => all features in train.csv
    ("longlist", "important_features_longlist_superconductivity.csv"),
    ("shortlist","important_features_shortlist_superconductivity.csv"),
]

all_summaries_all_datasets = []

for dataset_name, feature_file in dataset_specs:
    # Build this dataset's split
    X_train, X_val, X_verif, y_train, y_val, y_verif, used_features = make_feature_split(
        feature_file   = feature_file,
        main_data_file = 'train.csv',
        target_col     = 'critical_temp'
    )

    print(f"\n=== DATASET: {dataset_name} ===")
    print(f"Number of features used: {len(used_features)}")

    # Use the existing train/validation/verification split from earlier cells
    print("=== Using existing train/validation/verification split ===")
    print(f"Train size:          {X_train.shape[0]}")
    print(f"Validation size:     {X_val.shape[0]}")
    print(f"Verification size:   {X_verif.shape[0]}")
    print()

    # Standardize features (important for neural networks)
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train)
    X_val_scaled = scaler_X.transform(X_val)
    X_verif_scaled = scaler_X.transform(X_verif)

    # Reshape y for scaler (needs 2D array)
    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1)).flatten()
    y_val_scaled = scaler_y.transform(y_val.values.reshape(-1, 1)).flatten()
    y_verif_scaled = scaler_y.transform(y_verif.values.reshape(-1, 1)).flatten()

    print("=== Data standardization complete ===")
    print(f"X_train_scaled shape: {X_train_scaled.shape}")
    print(f"X_train_scaled mean (should be ~0): {X_train_scaled.mean():.6f}")
    print(f"X_train_scaled std (should be ~1): {X_train_scaled.std():.6f}")
    print()

    train_dataset = SuperconductorDataset(X_train_scaled, y_train_scaled)
    val_dataset = SuperconductorDataset(X_val_scaled, y_val_scaled)
    verif_dataset = SuperconductorDataset(X_verif_scaled, y_verif_scaled)

    all_summaries = []  # per-dataset
    input_size = X_train_scaled.shape[1]

    for cfg in experiment_configs:
        config_id     = cfg["config_id"]
        hidden_sizes  = cfg["hidden_sizes"]
        dropout_rate  = cfg["dropout"]
        learning_rate = cfg["learning_rate"]
        batch_size    = cfg["batch_size"]
        weight_decay  = cfg["weight_decay"]

        print("\n" + "=" * 80)
        print(f"=== Training config: {config_id} (dataset: {dataset_name}) ===")
        print(f"hidden_sizes={hidden_sizes}, dropout={dropout_rate}, "
              f"lr={learning_rate}, batch_size={batch_size}, weight_decay={weight_decay}")
        print("=" * 80 + "\n")

        # (re)create loaders for this config
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
        verif_loader = DataLoader(verif_dataset, batch_size=batch_size, shuffle=False)

        model = SimpleNN(input_size, hidden_sizes=hidden_sizes, dropout_rate=dropout_rate).to(device)

        print("=== Neural Network Architecture ===")
        print(model)
        print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
        print()

        # Loss function and optimizer
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

        # Training loop with comprehensive metric tracking
        num_epochs = 100
        metrics_history = {
            'epoch': [],
            'train_loss': [],
            'train_mse': [],
            'train_rmse': [],
            'train_r2': [],
            'val_loss': [],
            'val_mse': [],
            'val_rmse': [],
            'val_r2': [],
            'verif_loss': [],
            'verif_mse': [],
            'verif_rmse': [],
            'verif_r2': []
        }

        best_val_loss = float('inf')
        patience = 20
        patience_counter = 0

        print("=== Training Neural Network ===")
        for epoch in range(num_epochs):
            # Training phase
            model.train()
            train_loss = 0.0
            for X_batch, y_batch in train_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                
                optimizer.zero_grad()
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                optimizer.step()
                
                train_loss += loss.item()
            
            train_loss /= len(train_loader)
            
            # Evaluate on all sets (in original scale for RMSE and R²)
            train_metrics = evaluate_model_metrics(model, X_train_scaled, y_train, scaler_y, device)
            val_metrics   = evaluate_model_metrics(model, X_val_scaled,   y_val,   scaler_y, device)
            verif_metrics = evaluate_model_metrics(model, X_verif_scaled, y_verif, scaler_y, device)
            
            # Store metrics
            metrics_history['epoch'].append(epoch + 1)
            metrics_history['train_loss'].append(train_metrics['mse_scaled'])
            metrics_history['train_mse'].append(train_metrics['mse'])
            metrics_history['train_rmse'].append(train_metrics['rmse'])
            metrics_history['train_r2'].append(train_metrics['r2'])
            metrics_history['val_loss'].append(val_metrics['mse_scaled'])
            metrics_history['val_mse'].append(val_metrics['mse'])
            metrics_history['val_rmse'].append(val_metrics['rmse'])
            metrics_history['val_r2'].append(val_metrics['r2'])
            metrics_history['verif_loss'].append(verif_metrics['mse_scaled'])
            metrics_history['verif_mse'].append(verif_metrics['mse'])
            metrics_history['verif_rmse'].append(verif_metrics['rmse'])
            metrics_history['verif_r2'].append(verif_metrics['r2'])
            
            val_loss = val_metrics['mse_scaled']
            
            # Learning rate scheduling
            scheduler.step(val_loss)
            
            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                # Save best model (per config & per dataset)
                torch.save(model.state_dict(), f'best_nn_model_{dataset_name}_{config_id}.pth')
            else:
                patience_counter += 1
            
            if (epoch + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{num_epochs}]")
                print(f"  Train - Loss: {train_metrics['mse_scaled']:.6f}, RMSE: {train_metrics['rmse']:.4f} K, R²: {train_metrics['r2']:.4f}")
                print(f"  Val   - Loss: {val_metrics['mse_scaled']:.6f}, RMSE: {val_metrics['rmse']:.4f} K, R²: {val_metrics['r2']:.4f}")
                print(f"  Verif - Loss: {verif_metrics['mse_scaled']:.6f}, RMSE: {verif_metrics['rmse']:.4f} K, R²: {verif_metrics['r2']:.4f}")
            
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1} (config: {config_id}, dataset: {dataset_name})")
                break

        # Load best model
        model.load_state_dict(torch.load(f'best_nn_model_{dataset_name}_{config_id}.pth'))

        # Save metrics to CSV
        metrics_df = pd.DataFrame(metrics_history)
        csv_filename = f'nn_training_metrics_{dataset_name}_{config_id}.csv'
        metrics_df.to_csv(csv_filename, index=False)
        print(f"=== Saved training metrics to {csv_filename} ===")
        print(f"Metrics tracked for {len(metrics_history['epoch'])} epochs")
        print()

        # Create comprehensive plots (Verification set only)
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))

        # Plot 1: Loss curves (Verification only)
        axes[0, 0].plot(metrics_history['epoch'], metrics_history['verif_loss'], label='Verification Loss', linewidth=2, color='green')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('MSE Loss (scaled)')
        axes[0, 0].set_title(f'Neural Network: Verification Loss Curve ({config_id}, {dataset_name})')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)

        # Plot 2: RMSE curves (Verification only)
        axes[0, 1].plot(metrics_history['epoch'], metrics_history['verif_rmse'], label='Verification RMSE', linewidth=2, color='blue')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('RMSE (K)')
        axes[0, 1].set_title(f'Neural Network: Verification RMSE Curve ({config_id}, {dataset_name})')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)

        # Plot 3: R² curves (Verification only)
        axes[1, 0].plot(metrics_history['epoch'], metrics_history['verif_r2'], label='Verification R²', linewidth=2, color='red')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('R² Score')
        axes[1, 0].set_title(f'Neural Network: Verification R² Score Curve ({config_id}, {dataset_name})')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)

        # Plot 4: Combined view (RMSE and R²) - Verification only
        ax2 = axes[1, 1]
        ax2_twin = ax2.twinx()
        line1 = ax2.plot(metrics_history['epoch'], metrics_history['verif_rmse'], 'b-', label='Verif RMSE', linewidth=2)
        line2 = ax2_twin.plot(metrics_history['epoch'], metrics_history['verif_r2'], 'r-', label='Verif R²', linewidth=2)
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('RMSE (K)', color='b')
        ax2_twin.set_ylabel('R² Score', color='r')
        ax2.set_title(f'Neural Network: Verification Metrics (RMSE & R²) ({config_id}, {dataset_name})')
        ax2.tick_params(axis='y', labelcolor='b')
        ax2_twin.tick_params(axis='y', labelcolor='r')
        lines = line1 + line2
        labels = [l.get_label() for l in lines]
        ax2.legend(lines, labels, loc='center right')
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plot_filename = f'nn_training_curves_{dataset_name}_{config_id}.png'
        plt.savefig(plot_filename, dpi=300, bbox_inches='tight')
        print(f"=== Saved training curves to {plot_filename} ===")
        plt.show()

        # Evaluate on validation set (in original scale)
        model.eval()
        with torch.no_grad():
            X_val_tensor = torch.FloatTensor(X_val_scaled).to(device)
            y_val_pred_scaled = model(X_val_tensor).cpu().numpy()

        # Inverse transform predictions
        y_val_pred = scaler_y.inverse_transform(y_val_pred_scaled.reshape(-1, 1)).flatten()

        # Calculate metrics
        mse_nn = mean_squared_error(y_val, y_val_pred)
        rmse_nn = mse_nn ** 0.5
        r2_nn = r2_score(y_val, y_val_pred)

        print("\n=== Neural Network Performance on Validation Set ===")
        print(f"(config: {config_id}, dataset: {dataset_name})")
        print(f"Val MSE:  {mse_nn:.4f}")
        print(f"Val RMSE: {rmse_nn:.4f} K")
        print(f"Val R^2:  {r2_nn:.4f}")
        print()

        # Compare with previous models (if they exist)
        try:
            print("=== Model Comparison (Validation Set) ===")
            if 'rmse_rf' in globals() and 'r2_rf' in globals():
                print(f"Random Forest - RMSE: {rmse_rf:.4f} K, R²: {r2_rf:.4f}")
            if 'rmse_xgb' in globals() and 'r2_xgb' in globals():
                print(f"XGBoost      - RMSE: {rmse_xgb:.4f} K, R²: {r2_xgb:.4f}")
            print(f"Neural Net   - RMSE: {rmse_nn:.4f} K, R²: {r2_nn:.4f}")
            print()
        except:
            print("=== Neural Network Results (Validation Set) ===")
            print(f"RMSE: {rmse_nn:.4f} K")
            print(f"R²: {r2_nn:.4f}")
            print()

        # Evaluate on verification set (in original scale)
        model.eval()
        with torch.no_grad():
            X_verif_tensor = torch.FloatTensor(X_verif_scaled).to(device)
            y_verif_pred_scaled = model(X_verif_tensor).cpu().numpy()

        # Inverse transform predictions
        y_verif_pred = scaler_y.inverse_transform(y_verif_pred_scaled.reshape(-1, 1)).flatten()

        # Calculate metrics
        mse_verif = mean_squared_error(y_verif, y_verif_pred)
        rmse_verif = mse_verif ** 0.5
        r2_verif = r2_score(y_verif, y_verif_pred)

        print("=== Neural Network Performance on Verification Set ===")
        print(f"(config: {config_id}, dataset: {dataset_name})")
        print(f"Verif MSE:  {mse_verif:.4f}")
        print(f"Verif RMSE: {rmse_verif:.4f} K")
        print(f"Verif R^2:  {r2_verif:.4f}")
        print()

        # Create summary entry for comparison table
        best_epoch_idx = np.argmin(metrics_history['val_loss'])
        best_epoch = metrics_history['epoch'][best_epoch_idx]

        nn_summary = {
            'model_name': 'Neural Network',
            'dataset': dataset_name,
            'config_id': config_id,
            'train_mse': metrics_history['train_mse'][best_epoch_idx],
            'train_rmse': metrics_history['train_rmse'][best_epoch_idx],
            'train_r2': metrics_history['train_r2'][best_epoch_idx],
            'val_mse': metrics_history['val_mse'][best_epoch_idx],
            'val_rmse': metrics_history['val_rmse'][best_epoch_idx],
            'val_r2': metrics_history['val_r2'][best_epoch_idx],
            'verif_mse': metrics_history['verif_mse'][best_epoch_idx],
            'verif_rmse': metrics_history['verif_rmse'][best_epoch_idx],
            'verif_r2': metrics_history['verif_r2'][best_epoch_idx],
            'best_epoch': best_epoch,
            'hidden_layers': str(hidden_sizes),
            'dropout': dropout_rate,
            'learning_rate': learning_rate,
            'batch_size': batch_size,
            'weight_decay': weight_decay,
        }

        nn_summary_df = pd.DataFrame([nn_summary])
        summary_filename = f'nn_config_{dataset_name}_{config_id}_summary.csv'
        nn_summary_df.to_csv(summary_filename, index=False)
        print(f"=== Saved model summary to {summary_filename} ===")
        print("\nBest epoch summary:")
        print(nn_summary_df.to_string(index=False))

        all_summaries.append(nn_summary)

    # After all configs: combined comparison table for this dataset
    all_summaries_df = pd.DataFrame(all_summaries)
    all_summaries_df.to_csv(f'nn_all_configs_summary_{dataset_name}.csv', index=False)
    print(f"\n=== Saved all-configs summary to nn_all_configs_summary_{dataset_name}.csv ===")
    print(all_summaries_df.to_string(index=False))

    # accumulate across datasets
    for s in all_summaries:
        s['n_features'] = len(used_features)
    all_summaries_all_datasets.extend(all_summaries)

# After all datasets: combined comparison table
all_datasets_df = pd.DataFrame(all_summaries_all_datasets)
all_datasets_df.to_csv('nn_all_datasets_all_configs_summary.csv', index=False)
print("\n=== Saved all-datasets all-configs summary to nn_all_datasets_all_configs_summary.csv ===")
print(all_datasets_df.to_string(index=False))


In [ ]:

# Load summary file
df = pd.read_csv("/Users/Wei-Yang/Desktop/Berkeley/Studies/Courses/PHY 288/Project/Project3/Physics188_288_projectIII/data_eval/nn_all_datasets_all_configs_summary.csv")

# =========================
# styling function
# =========================
def styled_plot():
    plt.figure(figsize=(5,4))
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.xticks(rotation=45, fontweight='bold')
    plt.yticks(fontweight='bold')
    

# ================================
# Validation R2 comparison plot
# ================================
styled_plot()

for dataset in df["dataset"].unique():
    subset = df[df["dataset"] == dataset]
    plt.plot(subset["config_id"],
             subset["val_r2"],
             marker='o',
             linewidth=2,
             label=f"{dataset}")

plt.xlabel("Configuration", fontsize=12, fontweight='bold')
plt.ylabel("Validation R² Score", fontsize=12, fontweight='bold')
plt.title("Validation R² Comparison", fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()


# =================================
# 2) Verification R² comparison plot
# =================================
styled_plot()

for dataset in df["dataset"].unique():
    subset = df[df["dataset"] == dataset]
    plt.plot(subset["config_id"],
             subset["verif_r2"],
             marker='o',
             linewidth=2,
             label=f"{dataset}")

plt.xlabel("Configuration", fontsize=12, fontweight='bold')
plt.ylabel("Verification R² Score", fontsize=12, fontweight='bold')
plt.title("Verification R² Comparison", fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

plt.savefig("verif_r2_comparison.png", dpi=300, bbox_inches='tight')
plt.show()
